# PhysiCar SIM

Simulation HTTP API — served under `/sim/api/`, same layout as the physicar-sim README. Sections are ordered by how often you will reach for them.

## [0] Setup

In [1]:
import requests

BASE_URL = "http://localhost/sim/api"

# This API exists only in SIM mode — a real kit has no /sim/api
try:
    requests.get(f"{BASE_URL}/status", timeout=3).raise_for_status()
except requests.RequestException:
    raise SystemExit("Simulator not found — this notebook needs the PhysiCar simulator (SIM mode).")

## [1] Status & Time

### 1.1. Status

`GET /status` — simulator runtime status.

In [2]:
requests.get(f"{BASE_URL}/status").json()

{'running': True,
 'websocket': True,
 'current': 'custom_e09090b056ef1f90f845419690065271',
 'switching': False,
 'assets_version': 1786428741}

### 1.2. Clock

`GET /clock` — sim time / real time / RTF / paused.

In [ ]:
requests.get(f"{BASE_URL}/clock").json()

## [2] Vehicle

### 2.1. Pose

`GET /pose` — vehicle pose (world absolute coordinates).

In [ ]:
requests.get(f"{BASE_URL}/pose").json()

### 2.2. Teleport

`POST /pose` — `{"x": 1.0, "y": 2.0, "yaw": 0.0}`; omitted fields keep their current value. The pose is normalized upright at ground level, so this also rights a flipped car. The response returns after the pose is confirmed applied.

In [ ]:
pose = requests.get(f"{BASE_URL}/pose").json()
requests.post(f"{BASE_URL}/pose", json={"x": pose["x"] + 0.3}).json()

## [3] World & Objects

### 3.1. World

`GET /world` — one-call snapshot of the current world's immutable definition: identity (`world_id`/`rev`/display name), track geometry, object catalog, and whether it has an evaluation.

In [ ]:
world = requests.get(f"{BASE_URL}/world").json()
print({k: world[k] for k in ("world", "world_id", "rev", "display", "evaluation")})
print("track keys :", list(world["track"].keys()))
print("objects    :", list(world["objects"].keys()))

### 3.2. Route

`GET /route` — track centerline waypoints, plus inner/outer boundary lines when available.

In [ ]:
import matplotlib.pyplot as plt

route = requests.get(f"{BASE_URL}/route").json()

plt.figure(figsize=(6, 4))
for key, style in (("inner", "k-"), ("outer", "k-"), ("waypoints", "b--")):
    line = route.get(key)
    if line:
        xs, ys = zip(*line)
        plt.plot(xs, ys, style, linewidth=1)
plt.axis("equal")
plt.title(f"{len(route['waypoints'])} waypoints")
plt.show()

### 3.3. Bounds

`GET /bounds` — track bounding box.

In [ ]:
requests.get(f"{BASE_URL}/bounds").json()

### 3.4. Objects

`GET /objects` — world models (name, `type`: object/wall/light, static, movable, origin/current pose, size).

In [ ]:
objects = requests.get(f"{BASE_URL}/objects").json()["objects"]
objects

### 3.5. Move an object

`POST /models/<name>/pose` — `{"x", "y", "z", "yaw"}`; omitted fields keep their value, rotation is yaw-only. Works for World Builder objects and traffic lights; walls and the track itself are rejected. The response returns after the pose is confirmed applied.

In [ ]:
objects = requests.get(f"{BASE_URL}/objects").json()["objects"]
target = next(o for o in objects if o["movable"])
print("moving", target["name"])
requests.post(f"{BASE_URL}/models/{target['name']}/pose",
              json={"x": target["current"]["x"] + 0.2}).json()

## [4] Traffic Lights

### 4.1. List

`GET /traffic_lights` — the world's traffic lights and their states. Default state is `green`; states survive a respawn of the same world.

In [ ]:
lights = requests.get(f"{BASE_URL}/traffic_lights").json()["lights"]
lights

### 4.2. Change state

`POST /traffic_lights/<name>` — `{"state": "red"}` or `{"state": "green"}`. green→red passes through 3 s of yellow, during which commands are rejected with 409.

In [ ]:
import time

lights = requests.get(f"{BASE_URL}/traffic_lights").json()["lights"]
if lights:
    name = lights[0]["name"]
    print(requests.post(f"{BASE_URL}/traffic_lights/{name}", json={"state": "red"}).json())
    time.sleep(3.5)  # let the yellow transit finish
    print(requests.post(f"{BASE_URL}/traffic_lights/{name}", json={"state": "green"}).json())

## [5] Reset

### 5.1. Reset

`POST /reset` — every movable object, traffic light and the vehicle back at its start pose. Instant, pose-only, no world reload — the go-to reset between training episodes (this also undoes the teleport and object move above).

In [ ]:
requests.post(f"{BASE_URL}/reset").json()

### 5.2. Respawn

`POST /respawn` — reload the whole world (~6 s), the heavyweight reset for when the world misbehaves. Not executed here:

```python
requests.post(f"{BASE_URL}/respawn")
```

## [6] Display

### 6.1. Brightness

`GET /brightness`, `POST /brightness` — `{"value": 0.2..2.0}`, 1.0 = default. Applied instantly at the display layer (3D viewer + robot camera frames together); one shared server-side value, persists across world switches and restarts.

In [ ]:
print(requests.get(f"{BASE_URL}/brightness").json())
requests.post(f"{BASE_URL}/brightness", json={"value": 1.0}).json()

### 6.2. Overlay

`POST /overlay` — show status text on the `/sim` screen (`{"text": "...", "ttl": 10}` — text ≤300 chars, ttl 1–3600 s; expires by itself, e.g. training progress). `GET /overlay` reads the current text.

In [ ]:
requests.post(f"{BASE_URL}/overlay", json={"text": "Hello from the API", "ttl": 5}).json()

## [7] Monitoring

### 7.1. State

`GET /state` — one-call snapshot of everything that changes in real time: world/running/switching, sim `time`/`paused`/`rtf`, vehicle pose, object poses, traffic lights, overlay, brightness, and the evaluation run state.

In [ ]:
requests.get(f"{BASE_URL}/state").json()

### 7.2. Events (SSE)

`GET /events` — SSE stream of **named events**. `event: state` — full status snapshot pushed on change; `event: run` — student-process events during an evaluation. Consumers MUST ignore unknown event names.

In [ ]:
# Read the first pushed state event, then close the stream
with requests.get(f"{BASE_URL}/events", stream=True, timeout=10) as r:
    for line in r.iter_lines():
        print(line.decode()[:120])
        if line.startswith(b"data:"):
            break

## [8] Evaluation

`POST /evaluation/run` launches the student's code (`{"command"?, "time_limit_s"?}` — output streams to `event: run` on `/events`); `POST /evaluation/stop` stops it (idempotent). Normally driven by the ▶ button on the `/sim` page — not executed here.

### 8.1. Evaluation document

`GET /evaluation` — the current world's evaluation document (`{version, config, script}`), 404 when the world has none.

In [ ]:
r = requests.get(f"{BASE_URL}/evaluation")
print(r.status_code)
if r.ok:
    print(r.json()["config"])

## [9] Worlds

### 9.1. Worlds

`GET /worlds` — world list (includes the current one): `name`, `file`, `display`, `world_id`, `official`, `evaluation`, `deletable`.

In [ ]:
requests.get(f"{BASE_URL}/worlds").json()

### 9.2. Switch

`POST /switch` — switch world by file name or publish id; the id form resolves an installed published world and returns 404 when it is not installed. Takes several seconds (full world load) — not executed here:

```python
requests.post(f"{BASE_URL}/switch", json={"world": "physicar_base.world"})
# or by publish id:
requests.post(f"{BASE_URL}/switch", json={"world_id": "<32-hex id>"})
```

### 9.3. World publishing

`GET /worldpub` — current world's publish coordinates and the assets CDN revision. `POST /worlds/install` installs a published world from the CDN (`{"world_id": "<32-hex id>"}`, skipped with `"cached": true` when the same `rev` is already installed).

In [ ]:
requests.get(f"{BASE_URL}/worldpub").json()